In [ ]:
# Cell A (final revision): robust cleaning + improved name dedupe
import os, re, textwrap
from pathlib import Path
import numpy as np, pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_DESCRIPTORS = PROJECT_ROOT / "data" / "descriptors"
OUT = PROJECT_ROOT / "outputs" / "ml" / "random_forest"
OUT.mkdir(parents=True, exist_ok=True)

PS_IN = DATA_RAW / "ps_usable_hydrogen_storage_capacity_gcmcv2_real_cleaned.csv"
TPS_IN = DATA_RAW / "balanced_tps_canonical.csv"
REAL_IN = DATA_DESCRIPTORS / "mof_descriptors_12986.csv"

PS_OUT = OUT / "ps_cleaned_canonical.csv"
TPS_OUT = OUT / "tps_cleaned_canonical.csv"
REAL_OUT = OUT / "real_cleaned_canonical.csv"

CANONICAL = ["Density","PV","GSA","VSA","VF","PLD","LCD"]
NAME_CANDIDATES = ["Name","name","CSD Refcode","CSD_Refcode","csd_refcode","refcode","Refcode","ref_code","id","ID"]

def smart_read(path):
    for enc in ("utf-8-sig","utf-8","latin1","ISO-8859-1"):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    raise RuntimeError(f"Cannot read {path} with common encodings")

for p,lbl in [(PS_IN,"PS"),(TPS_IN,"TPS"),(REAL_IN,"REAL")]:
    if not Path(p).exists():
        raise FileNotFoundError(f"Missing input {lbl} at {p} - upload and re-run Cell A")

ps = smart_read(PS_IN); tps = smart_read(TPS_IN); real = smart_read(REAL_IN)

def normalize_cols(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df

ps = normalize_cols(ps); tps = normalize_cols(tps); real = normalize_cols(real)

ALIASES = {
    "Density": ["density","rho","framework_density","mass_density"],
    "PV":      ["pv","pore_volume","pore volume","porevolume","p_vol","pvol"],
    "GSA":     ["gsa","gravimetric_surface_area","surface_area_g","sa_g","m2_per_g","m2g","surfacearea_g"],
    "VSA":     ["vsa","volumetric_surface_area","surface_area_v","sa_v","m2_per_cm3","surfacearea_v"],
    "VF":      ["vf","void_fraction","porosity","porosity_fraction"],
    "PLD":     ["pld","pore_limiting_diameter","pore limiting diameter","pore_limit_diameter"],
    "LCD":     ["lcd","largest_cavity_diameter","largest cavity diameter","largest_cavity"]
}

def find_col_by_alias(df, alias_list):
    cols = list(df.columns)
    cols_low = [c.lower() for c in cols]
    for a in alias_list:
        for i,c in enumerate(cols_low):
            if a == c or a in c:
                return cols[i]
    for i,c in enumerate(cols_low):
        toks = re.split(r'[\s_\-]+', c)
        for a in alias_list:
            if a in toks:
                return cols[i]
    return None

def ensure_canonical(df):
    df = df.copy()
    for canon in CANONICAL:
        present = None
        for col in df.columns:
            if col.strip().lower() == canon.lower():
                present = col; break
        if present and present != canon:
            df = df.rename(columns={present: canon})
        elif canon not in df.columns:
            alias_list = ALIASES.get(canon, [canon.lower()])
            found = find_col_by_alias(df, [a.lower() for a in alias_list])
            if found:
                df = df.rename(columns={found: canon})
            else:
                df[canon] = np.nan
    for c in CANONICAL:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

ps = ensure_canonical(ps)
tps = ensure_canonical(tps)
real = ensure_canonical(real)

_SUFFIX_RE = re.compile(r'(?i)(?:_s[iI]_\d+|_inc\d+|_clean(ed)?|_core|_CoRE|_neutral|_charged|_manual|_clean|_k\d+|_100K|_11K|_40K|_70K|_si_\d+|_cm\d+|_cleaned)$')

def normalize_name_for_dedupe(name):
    if pd.isna(name):
        return ""
    s = str(name).strip()
    prev = None
    while prev != s:
        prev = s
        s = re.sub(_SUFFIX_RE, "", s).rstrip('_- ').strip()
    return s

def extract_base_and_suffix(name):

    s = normalize_name_for_dedupe(name)
    if s == "":
        return ("", -1)
    m = re.search(r'(\d+)$', s)
    if m:
        suffix = int(m.group(1))
        base = s[:m.start()].rstrip('_- ').strip()
        if base == "":
            base = s
        return (base, suffix)
    else:
        return (s, -1)

def dedupe_keep_highest_numeric(df, name_candidates=NAME_CANDIDATES, subset_by_desc=True, safety_threshold=0.60):
    df2 = df.copy().reset_index(drop=True)
    name_col = None
    for c in name_candidates:
        if c in df2.columns:
            name_col = c; break

    removed_by_name = 0
    removed_by_desc = 0
    kept_example = []
    dropped_example = []
    if name_col:
        parsed = df2[name_col].apply(lambda v: extract_base_and_suffix(v))
        df2["_dup_base"] = parsed.apply(lambda t: t[0])
        df2["_dup_num"]  = parsed.apply(lambda t: t[1])
        dup_counts = df2["_dup_base"].value_counts()
        multi_bases = dup_counts[dup_counts>1].index.tolist()
        if multi_bases:
            sample_base = multi_bases[:6]
            for b in sample_base:
                subset = df2[df2["_dup_base"]==b][[name_col,"_dup_num"]].astype(str)
                kept_example.append((b, subset.to_dict(orient="list")))
        # choose the index with maximum numeric suffix; if all -1 (no digits) will keep first occurrence
        idx_keep = df2.groupby("_dup_base")["_dup_num"].idxmax().values
        mask_keep = df2.index.isin(idx_keep)
        removed_by_name = int((~mask_keep).sum())
        # safety guard
        if removed_by_name / max(1, len(df2)) > safety_threshold:
            print(f"WARNING: name-based dedupe would remove {removed_by_name}/{len(df2)} rows (> {safety_threshold*100:.0f}%). Skipping name-based dedupe.")
            df2 = df2.drop(columns=["_dup_base","_dup_num"])
            removed_by_name = 0
        else:
            dropped_example = df2.loc[~mask_keep, name_col].astype(str).head(10).tolist()
            df2 = df2.loc[mask_keep].drop(columns=["_dup_base","_dup_num"]).reset_index(drop=True)

    # Deduplication
    if subset_by_desc:
        before_desc = len(df2)
        subset = [c for c in CANONICAL if c in df2.columns]
        if subset:
            df2 = df2.drop_duplicates(subset=subset).reset_index(drop=True)
            removed_by_desc = before_desc - len(df2)

    return df2.reset_index(drop=True), int(removed_by_name), int(removed_by_desc), kept_example, dropped_example

def drop_missing_desc(df):
    before = len(df)
    subset = [c for c in CANONICAL if c in df.columns]
    if not subset:
        return df, 0
    df2 = df.dropna(subset=subset).reset_index(drop=True)
    return df2, before - len(df2)

def apply_cleaning(df, label):
    before = len(df)
    candidate_name = None
    for c in NAME_CANDIDATES:
        if c in df.columns:
            candidate_name = c; break
    sample_names = df[candidate_name].dropna().astype(str).head(12).tolist() if candidate_name else []
    df_dedup, rem_name, rem_desc, kept_example, dropped_example = dedupe_keep_highest_numeric(df, name_candidates=NAME_CANDIDATES, subset_by_desc=True, safety_threshold=0.60)
    df_clean, rem_missing = drop_missing_desc(df_dedup)
    after = len(df_clean)
    print(textwrap.dedent(f"""
    --- {label} cleaning summary ---
      rows_before: {before}
      sample name column values (first 12) -> {sample_names if sample_names else 'no name-like column found'}
      removed_by_name_dedupe: {rem_name}
      removed_by_desc_dedupe: {rem_desc}
      removed_by_missing_desc (NaN in canonical): {rem_missing}
      rows_after: {after}
    """))
    if kept_example:
        print(" Example duplicate groups (kept highest suffix):")
        for b, info in kept_example:
            print(f"  base='{b}' -> entries: {info}")
    if dropped_example:
        print(" Example dropped names (some older suffixes):", dropped_example)
    if after == 0:
        safe_path = Path(OUT)/f"{label}_diagnostic_post_dedupe.csv"
        df_dedup.to_csv(safe_path, index=False)
        print(f" WARNING: {label} ended with 0 rows after cleaning. Intermediate dedup saved to {safe_path}")
    return df_clean

ps_clean = apply_cleaning(ps, "PS")
tps_clean = apply_cleaning(tps, "TPS")
real_clean = apply_cleaning(real, "REAL")

ps_clean.to_csv(PS_OUT, index=False)
tps_clean.to_csv(TPS_OUT, index=False)
real_clean.to_csv(REAL_OUT, index=False)

print("CELL A complete — cleaned files saved.")
print(f" PS -> {PS_OUT}  rows_before:{len(ps)}  rows_after:{len(ps_clean)}")
print(f" TPS -> {TPS_OUT} rows_before:{len(tps)}  rows_after:{len(tps_clean)}")
print(f" REAL -> {REAL_OUT} rows_before:{len(real)}  rows_after:{len(real_clean)}")
print(f"If rows_after==0 for any file, open the *_diagnostic_post_dedupe.csv in {OUT} to inspect intermediate dedupe.")

In [ ]:
# Cell B: RandomForestRegressor pipeline
RANDOM_STATE = 42
N_ITER_SEARCH = 12
CV_FOLDS = 10
TEST_SIZE = 0.25
HIGH_PERF_PERCENTILE = 90
MODEL_SLUG = "random_forest"
MODEL_LABEL = "RF"
RETRAIN_MODELS = False
USE_IDEAL_FEATURES = False
SAVE_TOPK = 200
N_JOBS = -1

import os, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, median_absolute_error, confusion_matrix
from scipy.stats import kendalltau
from sklearn.inspection import PartialDependenceDisplay
import joblib
warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

OUT = PROJECT_ROOT / "outputs" / "ml" / MODEL_SLUG
MODEL_DIR = PROJECT_ROOT / "models" / MODEL_SLUG
SPLIT_DIR = PROJECT_ROOT / "splits" / MODEL_SLUG
FIGURE_DIR = PROJECT_ROOT / "figures" / "ml" / MODEL_SLUG
for _path in [OUT, MODEL_DIR, SPLIT_DIR, FIGURE_DIR]:
    _path.mkdir(parents=True, exist_ok=True)

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_DESCRIPTORS = PROJECT_ROOT / "data" / "descriptors"

def resolve_clean_file(filename, fallback):
    candidate = OUT / filename
    return candidate if candidate.exists() else fallback

PS_CLEAN = resolve_clean_file("ps_cleaned_canonical.csv", DATA_PROCESSED / "ps_cleaned_canonical.csv")
TPS_CLEAN = resolve_clean_file("tps_cleaned_canonical.csv", DATA_PROCESSED / "tps_cleaned_canonical.csv")
REAL_CLEAN = resolve_clean_file("real_cleaned_canonical.csv", DATA_DESCRIPTORS / "mof_descriptors_12986.csv")

# Cleaned-file paths are resolved above from notebook outputs or repository data files.
for p in [PS_CLEAN, TPS_CLEAN, REAL_CLEAN]:
    if not p.exists():
        raise FileNotFoundError(f"Required cleaned file missing: {p}. Run Cell A first.")

ps = pd.read_csv(PS_CLEAN); tps = pd.read_csv(TPS_CLEAN); real = pd.read_csv(REAL_CLEAN)

CANONICAL = ["Density","PV","GSA","VSA","VF","PLD","LCD"]
for df,name in [(ps,"PS"),(tps,"TPS"),(real,"REAL")]:
    missing = [c for c in CANONICAL if c not in df.columns]
    if missing:
        raise KeyError(f"{name} cleaned file missing canonical descriptors: {missing}")

def detect_targets(df, prefix):
    cols = list(df.columns); cols_low = [c.lower() for c in cols]
    found = {}
    ug_alias = ["ug","usable_grav","usable_gravimetric","usable_g","gravimetric","wt%","g_h2_g","usable_hydrogen_gravimetric"]
    uv_alias = ["uv","usable_vol","usable_volumetric","usable_v","volumetric","g_h2_l","usable_hydrogen_volumetric"]
    for a in ug_alias:
        for i,c in enumerate(cols_low):
            if a in c:
                found[f"{prefix}_UG"] = cols[i]; break
        if f"{prefix}_UG" in found: break
    for a in uv_alias:
        for i,c in enumerate(cols_low):
            if a in c:
                found[f"{prefix}_UV"] = cols[i]; break
        if f"{prefix}_UV" in found: break
    return found

ps_targets = detect_targets(ps, "PS")
tps_targets = detect_targets(tps, "TPS")
print("Detected targets:", ps_targets, tps_targets)

tasks = {}
tasks.update(ps_targets); tasks.update(tps_targets)
if not tasks:
    raise RuntimeError("No labeled targets detected in PS/TPS cleaned files.")

if USE_IDEAL_FEATURES:
    extra = [f"{c}_dist_to_ideal" for c in CANONICAL] + ["meets_ideal_count","distance_score"]
    FEATURE_COLS = CANONICAL + extra
else:
    FEATURE_COLS = CANONICAL.copy()

param_dist = {
    "n_estimators": [100, 300, 500, 800],
    "max_depth": [None, 8, 12, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", 0.5, None],
    "bootstrap": [True, False]
}
cv_inner = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

def reg_metrics(y_true, y_pred):
    r2 = float(r2_score(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    medae = float(median_absolute_error(y_true, y_pred))
    try:
        kt = float(kendalltau(y_true, y_pred).correlation)
    except Exception:
        kt = None
    return {"r2": r2, "rmse": rmse, "mae": mae, "medae": medae, "kendall_tau": kt}

def class_metrics_from_threshold(y_true, y_pred, threshold):
    y_true_bin = (y_true >= threshold).astype(int)
    y_pred_bin = (y_pred >= threshold).astype(int)
    try:
        cm = confusion_matrix(y_true_bin, y_pred_bin)
    except Exception:
        cm = np.array([[0,0],[0,0]])
    if cm.size == 1:
        cm = np.array([[cm[0],0],[0,0]])
    tn, fp, fn, tp = int(cm[0,0]), int(cm[0,1]), int(cm[1,0]), int(cm[1,1])
    acc = (tn+tp)/(tn+tp+fp+fn) if (tn+tp+fp+fn)>0 else None
    prec = tp/(tp+fp) if (tp+fp)>0 else None
    rec = tp/(tp+fn) if (tp+fn)>0 else None
    f1 = (2*prec*rec/(prec+rec)) if (prec and rec and (prec+rec)>0) else None
    return {"confusion_tn": tn, "confusion_fp": fp, "confusion_fn": fn, "confusion_tp": tp,
            "accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

def precision_at_k(y_true, y_pred, k):
    n = len(y_true)
    if n==0: return None
    k = min(k, n)
    topk_pred = set(np.argsort(-np.array(y_pred))[:k])
    topk_true = set(np.argsort(-np.array(y_true))[:k])
    return float(len(topk_pred & topk_true)) / float(k)

results = {}
summary_rows = []
skipped_models = []

for task_key, target_col in tasks.items():
    print("\n" + "="*88)
    print(f"Task: {task_key}   (target: {target_col})")
    src = ps if task_key.startswith("PS") else tps
    dfw = src.dropna(subset=FEATURE_COLS + [target_col]).reset_index(drop=True)
    nrows = len(dfw)
    print(f" Labeled rows after dropna: {nrows}")
    if nrows < 10:
        msg = f"SKIP {task_key}: too few labeled rows ({nrows})"
        print(msg); skipped_models.append(msg); summary_rows.append({"task": task_key, "status": "skipped"})
        continue

    X = dfw[FEATURE_COLS].values
    y = pd.to_numeric(dfw[target_col], errors="coerce").values
    mask = ~np.isnan(y); X = X[mask]; y = y[mask]

    model_matrix_indices = np.arange(len(X))
    X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
        X, y, model_matrix_indices, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=True
    )
    split_records = pd.DataFrame({
        "split": ["train"] * len(train_idx) + ["test"] * len(test_idx),
        "model_matrix_index": np.concatenate([train_idx, test_idx]),
    })
    split_csv = SPLIT_DIR / f"{task_key}_rf_split_indices.csv"
    split_json = SPLIT_DIR / f"{task_key}_rf_split_manifest.json"
    split_records.to_csv(split_csv, index=False)
    with open(split_json, "w", encoding="utf-8") as f:
        json.dump({
            "task": task_key,
            "model": MODEL_LABEL,
            "target_column": target_col,
            "random_state": RANDOM_STATE,
            "test_size": TEST_SIZE,
            "shuffle": True,
            "feature_columns": FEATURE_COLS,
            "n_train": int(len(train_idx)),
            "n_test": int(len(test_idx)),
            "indices_csv": split_csv.name,
        }, f, indent=2)
    print(f" Train/test: {len(X_train)} / {len(X_test)}")

    best_model_path = MODEL_DIR / f"best_rf_{task_key}.joblib"
    final_model_path = MODEL_DIR / f"final_rf_{task_key}.joblib"
    if best_model_path.exists() and final_model_path.exists() and not RETRAIN_MODELS:
        best = joblib.load(best_model_path)
        best_params = {"loaded_from": str(best_model_path)}
        print(" Loaded saved tuned model:", best_model_path)
    else:
        base = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS)
        rs = RandomizedSearchCV(base, param_distributions=param_dist, n_iter=N_ITER_SEARCH, cv=cv_inner,
                                scoring="r2", n_jobs=N_JOBS, random_state=RANDOM_STATE, verbose=0)
        rs.fit(X_train, y_train)
        best = rs.best_estimator_
        best_params = rs.best_params_

    y_pred_test = best.predict(X_test)
    regm = reg_metrics(y_test, y_pred_test)
    thresh = np.percentile(y_train, HIGH_PERF_PERCENTILE)
    classm = class_metrics_from_threshold(y_test, y_pred_test, thresh)
    p3  = precision_at_k(y_test, y_pred_test, 3)
    p5  = precision_at_k(y_test, y_pred_test, 5)
    p10 = precision_at_k(y_test, y_pred_test, 10)
    p50 = precision_at_k(y_test, y_pred_test, 50)

    metrics = {**regm, **classm,
               "precision_at_3": p3, "precision_at_5": p5, "precision_at_10": p10, "precision_at_50": p50,
               "n_train": len(X_train), "n_test": len(X_test), "best_params": json.dumps(best_params)}
    metrics_path = OUT / f"metrics_{task_key}_RF.csv"
    pd.DataFrame([metrics]).to_csv(metrics_path, index=False)
    print(" Saved metrics:", metrics_path)

    print("\n Ahmed regression metrics:")
    print(f" R² = {regm['r2']:.4f}   RMSE = {regm['rmse']:.4f}   MAE = {regm['mae']:.4f}   MedianAE = {regm['medae']:.4f}")
    print(" Kendall's τ =", f"{regm['kendall_tau']:.4f}" if regm['kendall_tau'] is not None else "None")
    print("\n Classification diagnostics (threshold from TRAIN):")
    print(f" Threshold (train top {100-HIGH_PERF_PERCENTILE}%): {thresh:.6g}")
    print(f" Accuracy = {classm['accuracy']:.4f}   Precision = {classm['precision']:.4f}   Recall = {classm['recall']:.4f}   F1 = {classm['f1']:.4f}")
    print(f" Precision@3 = {p3}   Precision@5 = {p5}   Precision@10 = {p10}   Precision@50 = {p50}")
    print(" Best params (RandomizedSearchCV):", best_params)

    # parity
    plt.figure(figsize=(5,4)); plt.scatter(y_test, y_pred_test, s=18, alpha=0.6)
    mn, mx = min(np.nanmin(y_test), np.nanmin(y_pred_test)), max(np.nanmax(y_test), np.nanmax(y_pred_test))
    plt.plot([mn,mx],[mn,mx],"--", color="gray"); plt.xlabel("True"); plt.ylabel("Predicted"); plt.title(f"Parity: {task_key} (RF)")
    plt.tight_layout(); plt.savefig(OUT / f"parity_{task_key}_RF.png", dpi=150); plt.show(); plt.close()

    # confusion matrix
    cm = np.array([[classm["confusion_tn"], classm["confusion_fp"]],[classm["confusion_fn"], classm["confusion_tp"]]])
    pd.DataFrame(cm, index=["true_0","true_1"], columns=["pred_0","pred_1"]).to_csv(OUT / f"confusion_{task_key}_RF.csv")
    plt.figure(figsize=(3,3)); sns.heatmap(cm, annot=True, fmt="d", cmap="Blues"); plt.title(f"Confusion: {task_key} (RF)"); plt.tight_layout()
    plt.savefig(OUT / f"confusion_{task_key}_RF.png", dpi=150); plt.show(); plt.close()

    # feature importances
    try:
        fi = best.feature_importances_
        fi_df = pd.DataFrame({"feature": FEATURE_COLS, "importance": fi}).sort_values("importance", ascending=False)
        fi_df.to_csv(OUT / f"feature_importances_{task_key}_RF.csv", index=False)
    except Exception as e:
        print(" Feature importance save failed:", e)

    # SHAP (best-effort)
    try:
        import shap
        shap.initjs()
        sample_X = pd.DataFrame(X_train, columns=FEATURE_COLS).sample(n=min(500, len(X_train)), random_state=RANDOM_STATE)
        expl = shap.TreeExplainer(best)
        shap_vals = expl.shap_values(sample_X)
        shap.summary_plot(shap_vals, sample_X, show=False)
        plt.tight_layout(); plt.savefig(OUT / f"shap_{task_key}_RF.png", dpi=150); plt.close()
        print(" Saved SHAP:", OUT / f"shap_{task_key}_RF.png")
    except Exception as e:
        print(" SHAP unavailable/failed for", task_key, ":", e)

    # PDP
    try:
        top3 = fi_df.head(3)["feature"].tolist() if 'fi_df' in locals() else FEATURE_COLS[:3]
        Xtrain_df = pd.DataFrame(X_train, columns=FEATURE_COLS).dropna().head(400)
        if len(Xtrain_df) > 0:
            PartialDependenceDisplay.from_estimator(best, Xtrain_df, features=top3, kind="average")
            plt.tight_layout(); plt.savefig(OUT / f"pdp_{task_key}_RF_top3.png", dpi=150); plt.close()
            print(" Saved PDP:", OUT / f"pdp_{task_key}_RF_top3.png")
    except Exception as e:
        print(" PDP failed:", e)

    # Persist best and final models under models/<algorithm>/.
    if (not best_model_path.exists()) or RETRAIN_MODELS:
        joblib.dump(best, best_model_path)
    if final_model_path.exists() and not RETRAIN_MODELS:
        final = joblib.load(final_model_path)
        print(" Loaded saved final model:", final_model_path)
    else:
        final_params = best.get_params(); final_params["random_state"]=RANDOM_STATE; final_params["n_jobs"]=N_JOBS
        joblib.dump(final, final_model_path)
        print(" Saved final retrained RF model:", final_model_path)

    summary_rows.append({
        "task": task_key, "r2": regm["r2"], "rmse": regm["rmse"], "mae": regm["mae"], "medae": regm["medae"], "kendall_tau": regm["kendall_tau"],
        "precision_at_3": p3, "precision_at_5": p5, "precision_at_10": p10, "precision_at_50": p50,
        "n_train": len(X_train), "n_test": len(X_test), "best_params": json.dumps(best_params)
    })

    results[task_key] = {"model": final, "feature_cols": FEATURE_COLS}

# aggregated summary
summary_df = pd.DataFrame(summary_rows)
if not summary_df.empty:
    summary_df.to_csv(OUT / "aggregated_metrics_summary_rf.csv", index=False)
    print("\nAggregated per-target RF metrics saved to", OUT / "aggregated_metrics_summary_rf.csv")
else:
    print("No RF tasks completed.")

# predictions on REAL and composite ranking (unchanged)
real_preds = real.copy()
for c in FEATURE_COLS:
    if c not in real_preds.columns:
        real_preds[c] = np.nan
pred_cols = []
for task_key, info in results.items():
    model = info["model"]; featcols = info["feature_cols"]
    colname = f"ML_Predicted_{task_key}_RF"; pred_cols.append(colname); real_preds[colname]=np.nan
    mask_ok = ~real_preds[featcols].isna().any(axis=1)
    if mask_ok.sum()>0:
        real_preds.loc[mask_ok, colname] = model.predict(real_preds.loc[mask_ok, featcols].values)
    (real_preds[[colname]+featcols + (["OOD_flag"] if "OOD_flag" in real_preds.columns else [])]
     ).to_csv(OUT / f"ranking_{colname}.csv", index=False)

real_preds.to_csv(OUT / "REAL_with_MLpredictions_RF.csv", index=False)
pct_cols=[]
for c in pred_cols:
    pct=c+"_pctile"; real_preds[pct]=real_preds[c].rank(pct=True, method="average", na_option="keep"); pct_cols.append(pct)
if pct_cols:
    real_preds["mean_norm_rank"]=real_preds[pct_cols].mean(axis=1, skipna=True); real_preds["overall_rank"]=real_preds["mean_norm_rank"].rank(method="dense", ascending=False).astype("Int64")
else:
    real_preds["mean_norm_rank"]=np.nan; real_preds["overall_rank"]=pd.Series([pd.NA]*len(real_preds), dtype="Int64")
final_combined = real_preds.sort_values("mean_norm_rank", ascending=False)
final_combined.to_csv(OUT / "final_combined_ranking_RF.csv", index=False)
final_combined.head(SAVE_TOPK).to_csv(OUT / "top_200_overall_RF.csv", index=False)
final_combined.tail(SAVE_TOPK).to_csv(OUT / "bottom_200_overall_RF.csv", index=False)
print("Saved final combined ranking and top/bottom lists to", OUT)

In [ ]:
# Heatmap of Kendall's tau (models × targets)
# Optional summary figure for the repository output folder.

import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set(style="whitegrid")

# Kendall tau values extracted from your attached PDF
data = {
    "RF":       [0.8134, 0.7808, 0.9049, 0.8554],
    "ERT":      [0.8270, 0.7778, 0.9067, 0.8495],
    "LightGBM": [0.8209, 0.7722, 0.9073, 0.8545],
    "HistGB":   [0.8076, 0.7707, 0.9012, 0.8529],
    "SVR-RBF":  [0.8002, 0.7603, 0.8835, 0.8455],
    "KNN":      [0.8043, 0.7631, 0.8856, 0.8335]
}
targets = ["PS_UG","PS_UV","TPS_UG","TPS_UV"]
df = pd.DataFrame(data, index=targets).T  # rows=models, cols=targets

plt.figure(figsize=(9,4.2))
cmap = sns.color_palette("YlOrRd", as_cmap=True)
ax = sns.heatmap(df, annot=True, fmt=".3f", cmap=cmap, cbar_kws={'label':"Kendall's τ"}, vmin=0.75, vmax=0.92)
ax.set_title("Ranking fidelity (Kendall's τ) — models × targets")
ax.set_xlabel("Target"); ax.set_ylabel("Model")
plt.tight_layout()
out = FIGURE_DIR / "kendall_heatmap.png"
plt.savefig(out, dpi=200)
plt.show()
print("Saved:", out)

In [ ]:
# Combined panel: Test R² (left) and Test MedianAE (right)
# Optional summary figure for the repository output folder.
import os
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
sns.set(style="whitegrid", font_scale=1.0)

outpath = FIGURE_DIR / "r2_medae_panel.png"

models = ["RF","ERT","LightGBM","HistGB","SVR-RBF","KNN"]
targets = ["PS_UG","PS_UV","TPS_UG","TPS_UV"]

# <- numbers taken from your PDF paste (edit here if you want alternate values) ->
r2_vals = [
    [0.9838, 0.9319, 0.9891, 0.9367],   # RF
    [0.9863, 0.9331, 0.9905, 0.9315],   # ERT
    [0.9813, 0.9311, 0.9887, 0.9387],   # LightGBM
    [0.9847, 0.9305, 0.9883, 0.9373],   # HistGB
    [0.9789, 0.9310, 0.9861, 0.9277],   # SVR-RBF
    [0.9778, 0.9249, 0.9827, 0.9152],   # KNN
]

medae_vals = [
    [0.0803, 1.0365, 0.1076, 1.3855],  # RF  (note: UG@TPS medae=0.1076 per your new list)
    [0.0754, 1.0107, 0.1067, 1.3976],  # ERT
    [0.0788, 1.1169, 0.1051, 1.4073],  # LightGBM
    [0.0884, 1.0825, 0.1214, 1.4708],  # HistGB
    [0.0991, 1.0905, 0.1306, 1.4577],  # SVR-RBF
    [0.0721, 1.0495, 0.1229, 1.5886],  # KNN
]

r2_df = pd.DataFrame(r2_vals, index=models, columns=targets)
medae_df = pd.DataFrame(medae_vals, index=models, columns=targets)

# Plot combined figure
fig, axes = plt.subplots(1,2, figsize=(12,4.2), gridspec_kw={'width_ratios':[1,1]})
cmap1 = sns.color_palette("YlGnBu", as_cmap=True)
cmap2 = sns.color_palette("YlOrRd", as_cmap=True)

ax = axes[0]
sns.heatmap(r2_df, annot=True, fmt=".3f", cmap=cmap1, vmin=0.9, vmax=1.0, cbar_kws={'label':"Test R²"}, ax=ax)
ax.set_title("Test R² (models × targets)")
ax.set_xlabel("Target"); ax.set_ylabel("Model")

ax = axes[1]
sns.heatmap(medae_df, annot=True, fmt=".3f", cmap=cmap2, vmin=0.0, vmax=1.6, cbar_kws={'label':"Median Absolute Error"}, ax=ax)
ax.set_title("Test MedianAE (models × targets)")
ax.set_xlabel("Target"); ax.set_ylabel("")

plt.tight_layout()
plt.savefig(outpath, dpi=250)
plt.show()
print("Saved combined panel to:", outpath)